# Training-only resume from completed Drive tokens
Run cell 1 on CPU before allocating a GPU. On a fresh GPU connection rerun cell 1, then runtime setup, staging and the explicitly approved profile. No extraction or vision-model download exists here. The profile trains one epoch per arm (baseline, mean, patch), uses training validation only and never evaluates test retrieval. Checkpoints persist in a separate Drive output directory. Reconnect with the same pinned notebook to resume; existing incompatible checkpoints fail closed. Keep all approval flags false until reviewed.


In [ ]:
TRAINING_PREFLIGHT_OK = False
RESOURCE_SETUP_OK = False
STAGING_OK = False
CODE_REVISION = 'da4ffe964869a0c368a9b2136b575eca221dc41e'
import os, sys, json, subprocess, tempfile, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO=Path(tempfile.mkdtemp(prefix='training-only-',dir='/content'))/'repo'
subprocess.run(['git','clone','--no-checkout','https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',CODE_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_REVISION
WORK=Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP=WORK/'pilot_sub01_200concepts_v1'
OUT=WORK/'training_only_sub01_profile_v1'
CONFIG=REPO/'configs/patch_pilot.json'
sys.path.insert(0,str(REPO))
from phase1.process_log import run_logged
from phase1.patch_resources import BudgetMeter, runtime_profile
# Fresh child, GPU explicitly hidden. No array payload scan/model download.
args=[sys.executable,'-m','phase1.training_preflight','--dataset',str(PREP/'dataset.json'),
      '--config',str(CONFIG),'--output',str(OUT),'--staging-parent','/content']
previous_cuda=os.environ.get('CUDA_VISIBLE_DEVICES')
os.environ['CUDA_VISIBLE_DEVICES']=''
try:
    run_logged(args,REPO,WORK/'logs')
finally:
    if previous_cuda is None:os.environ.pop('CUDA_VISIBLE_DEVICES',None)
    else:os.environ['CUDA_VISIBLE_DEVICES']=previous_cuda
print('CPU training-input preflight passed. No training has run.')

TRAINING_PREFLIGHT_OK = True


In [ ]:
RESOURCE_SETUP_OK = False
assert globals().get('TRAINING_PREFLIGHT_OK',False), 'Run cell 1 (checkout and CPU input preflight) in this runtime first.'
APPROVE_GPU_SESSION = False #@param {type:"boolean"}
assert APPROVE_GPU_SESSION, 'Awaiting approval after CPU input verification.'
import time, os, json, subprocess, sys, uuid
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Connect a GPU runtime. No GPU model is guaranteed by Pro+.'
free,total = torch.cuda.mem_get_info()
GPU = {'name':torch.cuda.get_device_name(),'total_VRAM_GiB':total/2**30,'free_VRAM_GiB':free/2**30,
       'torch':torch.__version__,'cuda':torch.version.cuda,
       'precision':'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}
print(json.dumps(GPU,indent=2))
print('Last reported CU balance: 500; LIVE available CU and CU/hour: unknown until entered from Resources.')
# Colab: click the RAM/Disk resource indicator to view your balance and usage rate.
AVAILABLE_CU = 0.0 #@param {type:"number"}
CU_PER_HOUR = 0.0 #@param {type:"number"}
PILOT_CU_CAP = 25.0 #@param {type:"number"}
SESSION_HOUR_CAP = 2.0 #@param {type:"number"}
assert AVAILABLE_CU > 0 and CU_PER_HOUR > 0, 'Enter actual balance and CU/hour from Resources; do not guess.'
assert 0 < PILOT_CU_CAP <= AVAILABLE_CU and SESSION_HOUR_CAP > 0
print(f'Observed balance {AVAILABLE_CU:g} CU; observed rate {CU_PER_HOUR:g} CU/h.')
print(f'Pilot cap {PILOT_CU_CAP:g} CU; time allowed by cap {min(SESSION_HOUR_CAP,PILOT_CU_CAP/CU_PER_HOUR):.2f} h.')
SESSION_START = time.time()
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/eeg_results/patch_phase1')
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
PREP = WORK/'pilot_sub01_200concepts_v1'
TOKEN_DIR = PREP/'tokens_L28'
PREPARED_INPUTS_CHECKED = False
WORK.mkdir(parents=True,exist_ok=True)
LEDGER = WORK/'pilot_resource_usage.jsonl'
prior_rows = [json.loads(line) for line in LEDGER.read_text().splitlines() if line.strip()] if LEDGER.exists() else []
PRIOR_CU = max([r.get('estimated_cu') or 0 for r in prior_rows],default=0.)
assert PRIOR_CU < PILOT_CU_CAP, 'Previous pilot sessions reached the cap; review the ledger before increasing it.'
os.environ.update(PHASE1_SESSION_START=str(SESSION_START),PHASE1_CU_RATE=str(CU_PER_HOUR),
                  PHASE1_CU_LIMIT=str(PILOT_CU_CAP),PHASE1_CU_PRIOR=str(PRIOR_CU),
                  PHASE1_SECONDS_LIMIT=str(SESSION_HOUR_CAP*3600),PHASE1_RESOURCE_LOG=str(LEDGER))
SESSION = uuid.uuid4().hex[:8]
(WORK/f'runtime_{SESSION}.json').write_text(json.dumps(dict(**GPU,available_CU_observed=AVAILABLE_CU,
    CU_per_hour_observed=CU_PER_HOUR,observed_at=SESSION_START,prior_pilot_CU_estimate=PRIOR_CU),indent=2))
# Import where used; do not rely on symbols retained from a previous cell.
from phase1.patch_resources import BudgetMeter, runtime_profile
from phase1.process_log import run_logged
meter=BudgetMeter()
print(runtime_profile())
with open(WORK/f'training_environment_{SESSION}.txt','w') as f:
    subprocess.run([sys.executable,'-m','pip','freeze'],stdout=f,check=True)
def monitored(args):
    return run_logged(args,REPO,WORK/'logs',check_budget=lambda:meter.check('training_only',force=True))

RESOURCE_SETUP_OK = True


In [ ]:
STAGING_OK = False
assert globals().get('RESOURCE_SETUP_OK',False), 'Run cell 2 (GPU resources) successfully first.'
import tempfile, shutil
from pathlib import Path
# Recheck with CPU operations, then stage only manifest-listed inputs.
from phase1.training_preflight import check
from phase1.patch_cache import TokenCache
info=check(PREP/'dataset.json',CONFIG,OUT,'/content')
LOCAL=Path(tempfile.mkdtemp(prefix='training-data-',dir='/content'))
for name in info['files']:
    meter.check('staging',force=True)
    source=PREP/name;target=LOCAL/name
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(source,target)
    assert source.stat().st_size==target.stat().st_size, f'Incomplete copy: {name}'
TokenCache(LOCAL/'tokens_L28',LOCAL/'catalog.json')
(LOCAL/'STAGING_COMPLETE').write_text(str(PREP))
print('Staged completed inputs:',LOCAL)

STAGING_OK = True


In [ ]:
assert globals().get('STAGING_OK',False), 'Complete cell 3 (staging) in this runtime first.'
APPROVE_ONE_EPOCH_PROFILE = False #@param {type:"boolean"}
assert APPROVE_ONE_EPOCH_PROFILE, 'Approve only one epoch per arm after preflight and budget review.'
assert (LOCAL/'STAGING_COMPLETE').is_file()
args=[sys.executable,'-m','phase1.patch_experiment','--dataset',str(LOCAL/'dataset.json'),
      '--config',str(CONFIG),'--output',str(OUT),'--profile-one-epoch']
if OUT.exists():args+=['--resume']
monitored(args)
assert (OUT/'PILOT_PROFILE_COMPLETE').is_file()
assert not (OUT/'results.csv').exists(), 'Unexpected test results; inspect output provenance.'
print('One-epoch profile complete. Checkpoints:',OUT)
print((OUT/'epoch_timing.jsonl').read_text())
print('STOP. No full-training or test-evaluation command is provided. Disconnect to avoid idle CU use.')
